# PriViLege (CVPR 2024): reproduction on Colab

Runs the official PriViLege code (branch `colab-fixes` of `AkilaDissanayake/PriViLege`) on Colab.

**Before you start**
1. Pick an **A100** or **L4** GPU. A T4 (16 GB) will likely run out of memory at batch 128.
   - *Colab in the browser:* Runtime -> Change runtime type.
   - *VS Code (Colab extension):* Select Kernel (top right) -> Colab -> **New Colab Server** -> GPU -> A100 or L4. "Auto Connect" gives a CPU server.
2. Datasets download automatically the first time they are needed and are kept in `MyDrive/PriViLege/datasets/` (CUB200 ~1.1 GB and miniImageNet from the CEC Google Drive folder; CIFAR-100 from Toronto). Later sessions reuse the copies in Drive.
3. Run cells 1-6 at the start of **every** Colab session. Then run the experiment cells you need.
4. *VS Code:* press Ctrl+S after a cell finishes, so its output is saved in the .ipynb file.

**If anything stops** (disconnect, runtime reset, you press Stop): run cells 1-6 again, then re-run the same experiment cell.
Finished runs are skipped. The interrupted run continues from its last checkpoint, which is saved to Drive (`MyDrive/PriViLege/resume/`) after every base epoch and every incremental session.
Run the **Status** cell (6b) at any time to see what is finished, resumable or not started.

## 1. Drive, GPU and paths
**Why:** Drive keeps datasets, logs and results after the Colab VM is wiped. Training data goes on the fast local disk (`/content/data`).

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/PriViLege'   # persistent: datasets/, logs/, results/
DATA  = '/content/data'                      # fast local disk for training data
REPO  = '/content/PriViLege'
for d in ['datasets', 'logs', 'results']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
os.makedirs(DATA, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/bin/bash: line 1: nvidia-smi: command not found


## 2. Get the patched code
**Why:** the `colab-fixes` branch has the three one-line fixes (pandas 2, miniImageNet map path, unused `open_clip` import). The commit id goes into the results log so every number can be traced to the exact code.

In [14]:
if not os.path.exists(REPO):
    !git clone -b colab-fixes https://github.com/AkilaDissanayake/PriViLege.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
!git log -1 --oneline

Already up to date.
/content/PriViLege
dee2953 (HEAD -> colab-fixes, origin/colab-fixes) Auto-download missing datasets to Drive


## 3. Install only what is missing
**Why:** do **not** install `requirements.txt` (a full Anaconda dump that breaks Colab). Colab already has torch, transformers, pandas and so on. Only timm is pinned to **0.6.13**. It loads the same `vit_base_patch16_224` checkpoint as the authors' 0.6.12, and 0.6.12 itself fails to import on Colab's Python 3.12. timm 0.9 and newer load a different checkpoint under the same name.

In [15]:
!pip install -q timm==0.6.13
import timm, torch, transformers, pandas
print('timm', timm.__version__, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| pandas', pandas.__version__)
assert timm.__version__ == '0.6.13', 'Restart the kernel (browser: Runtime -> Restart session; VS Code: Restart button) and run cells 1-3 again'
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('GPU:', GPU)

timm 0.6.13 | torch 2.14.1+cpu | transformers 5.22.1 | pandas 2.2.3
GPU: none


## 4. Dataset helpers
**Why:** copying one archive from Drive and extracting it locally takes seconds or minutes. Reading thousands of small JPEGs directly from Drive would make every epoch many times slower. `prepare()` is called automatically by `run()`.

In [16]:
import shutil

# Google Drive file ids of the archives in the CEC folder used by FSCIL papers
# https://drive.google.com/drive/folders/11LxZCQj2FRCs0JTsf_dafvTHqFn2yGSN
GDRIVE_IDS = {
    'cub200':        '1EtKb1Bxd9ZtK-WUFk5dIhoOdhLeohA1q',   # CUB_200_2011.tgz
    'mini_imagenet': '1_x4o0iFetEv-T3PeIxdSbBPUG4hFfT8U',   # miniimagenet.tar
}

ARCHIVES = {   # dataset -> (archive in Drive, folder after extraction)
    'cifar100':      ('cifar-100-python.tar.gz', 'cifar-100-python'),
    'cub200':        ('CUB_200_2011.tgz',        'CUB_200_2011'),
    'mini_imagenet': ('miniimagenet.tar',        'miniimagenet'),
}

def prepare(dataset):
    archive, folder = ARCHIVES[dataset]
    src = f'{DRIVE}/datasets/{archive}'
    if os.path.isdir(f'{DATA}/{folder}'):
        pass
    elif os.path.exists(src):
        print(f'Extracting {archive} ...')
        get_ipython().system(f'tar -xf "{src}" -C {DATA}')
    elif dataset == 'cifar100':
        print('CIFAR-100 is not in Drive yet; train.py will download it (saved to Drive afterwards).')
    else:
        download_to_drive(dataset)
        print(f'Extracting {archive} ...')
        get_ipython().system(f'tar -xf "{src}" -C {DATA}')
    if dataset == 'mini_imagenet' and not os.path.exists(f'{DATA}/miniimagenet/split/train.csv'):
        os.makedirs(f'{DATA}/miniimagenet/split', exist_ok=True)   # same csv files ship with the repo
        get_ipython().system(f'cp {REPO}/data/index_list/mini_imagenet/*.csv {DATA}/miniimagenet/split/')

def download_to_drive(dataset):
    # Download a missing archive once: to local disk first (fast), then copy into Drive
    import gdown
    archive = ARCHIVES[dataset][0]
    tmp, dst = f'/content/{archive}', f'{DRIVE}/datasets/{archive}'
    print(f'{archive} is not in Drive: downloading it from the CEC Google Drive folder ...')
    gdown.download(id=GDRIVE_IDS[dataset], output=tmp, quiet=False)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 10_000_000:
        raise RuntimeError(f'Download of {archive} failed (Google Drive quota?). Try again later, or put it in '
                           f'{DRIVE}/datasets/ by hand from the CEC folder.')
    shutil.copy(tmp, dst + '.part'); os.replace(dst + '.part', dst); os.remove(tmp)
    print(f'Saved {archive} to {dst} ({os.path.getsize(dst) / 1e9:.2f} GB)')

def save_cifar_to_drive():
    src, dst = f'{DATA}/cifar-100-python.tar.gz', f'{DRIVE}/datasets/cifar-100-python.tar.gz'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print('Saved CIFAR-100 archive to Drive')

def check_layout(dataset):
    need = {
        'cifar100':      ['cifar-100-python/train', 'cifar-100-python/test', 'cifar-100-python/meta'],
        'cub200':        ['CUB_200_2011/images.txt', 'CUB_200_2011/train_test_split.txt',
                          'CUB_200_2011/image_class_labels.txt', 'CUB_200_2011/classes.txt',
                          'CUB_200_2011/images'],
        'mini_imagenet': ['miniimagenet/images', 'miniimagenet/split/train.csv', 'miniimagenet/split/test.csv'],
    }[dataset]
    missing = [p for p in need if not os.path.exists(f'{DATA}/{p}')]
    print(dataset, 'OK' if not missing else f'MISSING: {missing}')

## 5. Check the pretrained weights
**Why:** confirms you get the authors' backbone (ImageNet-21k pre-trained, ImageNet-1k fine-tuned AugReg ViT-B/16). It also caches the ViT and BERT downloads, so a download problem shows up now instead of mid-run.

In [17]:
m = timm.create_model('vit_base_patch16_224', pretrained=True)
print(m.default_cfg['url'])   # expect .../augreg/B_16-i21k-300ep-...--imagenet2012-...res_224.npz
assert 'i21k' in m.default_cfg['url']
del m
from transformers import BertTokenizer, BertModel
BertTokenizer.from_pretrained('bert-base-cased'); BertModel.from_pretrained('bert-base-cased')
print('ViT-B/16 and bert-base-cased ready')

https://storage.googleapis.com/vit_models/augreg/B_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.0-sd_0.0--imagenet2012-steps_20k-lr_0.01-res_224.npz


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ViT-B/16 and bert-base-cased ready


## 6. `run()`, `status()` and `summarize()`
**Why:** `run()` uses the official `run_script/vit_run_pretrain.py` arguments unchanged, so ~40 runs have no typos. It also:
- **skips** a run that is already in `results/results_log.csv`, so re-running a loop continues where it stopped;
- **resumes** an interrupted run: `-ckpt_dir` makes `train.py` save its state to `MyDrive/PriViLege/resume/<out>/<dataset>/seed<k>/` after every base epoch and every session, and continue from there on restart (the state is deleted once the run finishes);
- appends the log to Drive (`tee -a`), so nothing is lost on a disconnect;
- copies only `results.txt` to Drive (each run also writes two ~350 MB `.pth` files);
- adds one row to `results_log.csv` (dataset, seed, GPU, commit, A_Base / A_Last / A_Avg, minutes, resumed).

Every ablation needs its own `out` name, because the save path doesn't include the `-ED`/`-SKD`/`-taskblock` flags.
Use `run(..., force=True)` to redo a finished run from scratch.

In [18]:
import ast, csv, glob, json, time, statistics as st, collections

LOG_CSV = f'{DRIVE}/results/results_log.csv'
RESUME  = f'{DRIVE}/resume'

def parse_results(path):
    accs = ast.literal_eval(open(path).read().strip().splitlines()[-1])   # per-session accuracies
    return accs[0], accs[-1], sum(accs) / len(accs)                       # A_Base, A_Last, A_Avg

def finished_runs():
    if not os.path.exists(LOG_CSV):
        return {}
    with open(LOG_CSV, newline='') as f:
        return {(r['dataset'], int(r['seed']), r['out']): r for r in csv.DictReader(f)}

def describe_state(path):
    s = json.load(open(path))
    where = (f"base epoch {s['epoch'] + 1}/{s['epochs_base']}" if s['phase'] == 'base'
             else f"session {s['session']}/{s['sessions'] - 1}")
    return f"saved after {where} at {s['saved_at']}"

def run(dataset, seed, out='PriViLege', losses='-ED -SKD', extra='',
        epochs_base=5, epochs_new=3, batch_size=128, force=False):
    done = finished_runs().get((dataset, seed, out))
    ckpt = f'{RESUME}/{out}/{dataset}/seed{seed}'
    if done and not force:
        print(f"== {dataset} seed {seed} ({out}) already finished: A_Base {done['A_Base']} | "
              f"A_Last {done['A_Last']} | A_Avg {done['A_Avg']}. Skipping.")
        return float(done['A_Base']), float(done['A_Last']), float(done['A_Avg'])
    if force:
        shutil.rmtree(ckpt, ignore_errors=True)
    prepare(dataset)
    save_cifar_to_drive() if dataset == 'cifar100' else None
    state_json = f'{ckpt}/resume_state.json'
    resumed = os.path.exists(state_json)
    print(f'>> {dataset} seed {seed} ({out}): ' + (f'RESUMING, {describe_state(state_json)}' if resumed else 'starting'))
    log = f'{DRIVE}/logs/{dataset}_{out}_seed{seed}.log'
    cmd = (f'python train.py -project base -dataset {dataset} -base_mode ft_dot '
           f'-new_mode avg_cos -gamma 0.1 -lr_base 2e-4 -lr_new 2e-4 -decay 0.0005 '
           f'-epochs_base {epochs_base} -epochs_new {epochs_new} -schedule Cosine '
           f'-milestones 20 30 45 -gpu 0 -temperature 16 -start_session 0 '
           f'-batch_size_base {batch_size} -seed {seed} -vit -comp_out 1 -LT {losses} {extra} '
           f'-out {out} -dataroot {DATA} -ckpt_dir {ckpt}')
    print(cmd, '\n')
    with open(log, 'a') as f:
        f.write(f"\n===== {'resume' if resumed else 'start'} {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    t0 = time.time()
    get_ipython().system(f'cd {REPO} && {cmd} 2>&1 | tee -a {log}')
    minutes = (time.time() - t0) / 60
    if dataset == 'cifar100':
        save_cifar_to_drive()
    if os.path.isdir(f'{REPO}/checkpoint'):
        get_ipython().system(f"rsync -a --exclude='*.pth' {REPO}/checkpoint/ {DRIVE}/results/")
    hits = [p for p in glob.glob(f'{REPO}/checkpoint/{out}/{dataset}/**/*seed_{seed}-*/results.txt', recursive=True)
            if os.path.getmtime(p) > t0]
    if not hits:
        msg = describe_state(state_json) if os.path.exists(state_json) else 'no checkpoint saved yet'
        print(f'\n!! Run stopped before finishing ({msg}). Re-run this cell to continue. Log: {log}')
        return None
    a_base, a_last, a_avg = parse_results(hits[0])
    new_file = not os.path.exists(LOG_CSV)
    with open(LOG_CSV, 'a', newline='') as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(['dataset', 'seed', 'out', 'flags', 'batch', 'gpu', 'commit',
                        'A_Base', 'A_Last', 'A_Avg', 'minutes', 'resumed', 'date'])
        w.writerow([dataset, seed, out, f'{losses} {extra}'.strip(), batch_size, GPU, COMMIT,
                    f'{a_base:.2f}', f'{a_last:.2f}', f'{a_avg:.2f}', f'{minutes:.1f}',
                    'yes' if resumed else 'no', time.strftime('%Y-%m-%d %H:%M')])
    shutil.rmtree(ckpt, ignore_errors=True)   # run finished: free the Drive space of its resume state
    print(f'\n== {dataset} seed {seed} ({out}): A_Base {a_base:.2f} | A_Last {a_last:.2f} | A_Avg {a_avg:.2f} | {minutes:.1f} min')
    return a_base, a_last, a_avg

PLAN = ([('cifar100', 1, 'smoke')] +
        [(ds, s, 'PriViLege') for ds in ['cifar100', 'cub200', 'mini_imagenet'] for s in [1, 2, 3, 4, 5]] +
        [('cub200', s, o) for o in ['abl_PKT', 'abl_PKT_ED', 'abl_PKT_SKD'] for s in [1, 2, 3, 4, 5]] +
        [('cub200', 1, f'layers_{L}') for L in [0, 2, 5, 7, 10, 12]])

def status(plan=PLAN):
    done = finished_runs()
    counts = collections.Counter()
    for ds, s, out in plan:
        state_json = f'{RESUME}/{out}/{ds}/seed{s}/resume_state.json'
        if (ds, s, out) in done:
            r = done[(ds, s, out)]
            msg = f"done        A_Base {r['A_Base']}  A_Last {r['A_Last']}  A_Avg {r['A_Avg']}"
            counts['done'] += 1
        elif os.path.exists(state_json):
            msg = f'RESUMABLE   {describe_state(state_json)}'
            counts['resumable'] += 1
        else:
            msg = 'not started'
            counts['not started'] += 1
        print(f'{ds:14s} seed {s}  {out:12s} {msg}')
    print('\n' + ', '.join(f'{k}: {v}' for k, v in counts.items()))

def summarize(out='PriViLege'):
    by = collections.defaultdict(list)
    for p in sorted(glob.glob(f'{DRIVE}/results/{out}/**/results.txt', recursive=True)):
        ds = p.split(f'/{out}/')[1].split('/')[0]
        seed = p.split('seed_')[1].split('-')[0]
        r = parse_results(p)
        by[ds].append(r)
        print(f'{ds:14s} seed {seed}: Base {r[0]:.2f}  Last {r[1]:.2f}  Avg {r[2]:.2f}')
    for ds, v in by.items():
        cols = list(zip(*v))
        fmt = lambda xs: f'{st.mean(xs):.2f} +/- {st.stdev(xs) if len(xs) > 1 else 0:.2f}'
        print(f'== {ds} (n={len(v)}): A_Base {fmt(cols[0])} | A_Last {fmt(cols[1])} | A_Avg {fmt(cols[2])}')

## 6b. Status: what is finished, resumable or not started
**Why:** after a disconnect this shows where everything stands, using `results_log.csv` and the resume states in Drive.

In [19]:
status()

cifar100       seed 1  smoke        not started
cifar100       seed 1  PriViLege    not started
cifar100       seed 2  PriViLege    not started
cifar100       seed 3  PriViLege    not started
cifar100       seed 4  PriViLege    not started
cifar100       seed 5  PriViLege    not started
cub200         seed 1  PriViLege    not started
cub200         seed 2  PriViLege    not started
cub200         seed 3  PriViLege    not started
cub200         seed 4  PriViLege    not started
cub200         seed 5  PriViLege    not started
mini_imagenet  seed 1  PriViLege    not started
mini_imagenet  seed 2  PriViLege    not started
mini_imagenet  seed 3  PriViLege    not started
mini_imagenet  seed 4  PriViLege    not started
mini_imagenet  seed 5  PriViLege    not started
cub200         seed 1  abl_PKT      not started
cub200         seed 2  abl_PKT      not started
cub200         seed 3  abl_PKT      not started
cub200         seed 4  abl_PKT      not started
cub200         seed 5  abl_PKT      not 

## 7. Smoke test (1 base epoch + 1 epoch per session, ~10-20 min)
**Why:** this catches environment, data and patch problems before you spend time on full runs. The pandas crash fixed in `colab-fixes` would only appear after the base session, so a pass means all 9 sessions run.

It passes if the log shows `Incremental session: [1]` ... `[8]` and the final line prints A_Base / A_Last / A_Avg. Smoke-test numbers are not meaningful.

In [20]:
run('cifar100', seed=1, out='smoke', epochs_base=1, epochs_new=1)

CIFAR-100 is not in Drive yet; train.py will download it (saved to Drive afterwards).
>> cifar100 seed 1 (smoke): starting
python train.py -project base -dataset cifar100 -base_mode ft_dot -new_mode avg_cos -gamma 0.1 -lr_base 2e-4 -lr_new 2e-4 -decay 0.0005 -epochs_base 1 -epochs_new 1 -schedule Cosine -milestones 20 30 45 -gpu 0 -temperature 16 -start_session 0 -batch_size_base 128 -seed 1 -vit -comp_out 1 -LT -ED -SKD  -out smoke -dataroot /content/data -ckpt_dir /content/drive/MyDrive/PriViLege/resume/smoke/cifar100/seed1 

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
GPU COUNT 0
Devices: []
Device Name: []
manual seed: 1
{'ED': True,
 'ED_hp': 0.1,
 'LT': True,
 'MP': False,
 'PKT_tune_way': 1,
 'SKD': True,
 'WC': False,
 'base_mode': 'ft_dot',
 'baseline': False,
 'batch_size_base': 128,
 'batch_size_new': 0,
 'ckpt_dir': '/content/drive/MyD

## 8. Main reproduction runs (paper Table 1)
**Why 5 seeds:** the paper reports mean +/- std over 5 runs. **Why this order:** CIFAR-100 needs no data preparation. CUB200 is quick and is also used for the ablations. miniImageNet has the most data.

Paper targets (A_Base / A_Last / A_Avg):

| Dataset | A_Base | A_Last | A_Avg |
|---|---|---|---|
| CIFAR-100 | 90.88 +/- 0.20 | 86.06 +/- 0.32 | 88.08 +/- 0.20 |
| CUB200 | 82.21 +/- 0.35 | 75.08 +/- 0.52 | 77.50 +/- 0.33 |
| miniImageNet | 96.68 +/- 0.06 | 94.10 +/- 0.13 | 95.27 +/- 0.11 |

If Colab stops, run cells 1-6 again and re-run the same cell: finished seeds are skipped and the interrupted seed resumes from its last checkpoint.

In [21]:
for seed in [1, 2, 3, 4, 5]:
    run('cifar100', seed)

CIFAR-100 is not in Drive yet; train.py will download it (saved to Drive afterwards).
>> cifar100 seed 1 (PriViLege): starting
python train.py -project base -dataset cifar100 -base_mode ft_dot -new_mode avg_cos -gamma 0.1 -lr_base 2e-4 -lr_new 2e-4 -decay 0.0005 -epochs_base 5 -epochs_new 3 -schedule Cosine -milestones 20 30 45 -gpu 0 -temperature 16 -start_session 0 -batch_size_base 128 -seed 1 -vit -comp_out 1 -LT -ED -SKD  -out PriViLege -dataroot /content/data -ckpt_dir /content/drive/MyDrive/PriViLege/resume/PriViLege/cifar100/seed1 

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
GPU COUNT 0
Devices: []
Device Name: []
manual seed: 1
{'ED': True,
 'ED_hp': 0.1,
 'LT': True,
 'MP': False,
 'PKT_tune_way': 1,
 'SKD': True,
 'WC': False,
 'base_mode': 'ft_dot',
 'baseline': False,
 'batch_size_base': 128,
 'batch_size_new': 0,
 'ckpt_dir': '/conte

In [22]:
check_layout('cub200') if os.path.isdir(f'{DATA}/CUB_200_2011') else prepare('cub200')
for seed in [1, 2, 3, 4, 5]:
    run('cub200', seed)

CUB_200_2011.tgz is not in Drive: downloading it from the CEC Google Drive folder ...


Downloading...
From (original): https://drive.google.com/uc?id=1EtKb1Bxd9ZtK-WUFk5dIhoOdhLeohA1q
From (redirected): https://drive.google.com/uc?id=1EtKb1Bxd9ZtK-WUFk5dIhoOdhLeohA1q&confirm=t&uuid=4b997595-ba86-460c-9f05-56e478045988
To: /content/CUB_200_2011.tgz
100%|██████████| 1.15G/1.15G [00:26<00:00, 42.9MB/s]


Saved CUB_200_2011.tgz to /content/drive/MyDrive/PriViLege/datasets/CUB_200_2011.tgz (1.15 GB)
Extracting CUB_200_2011.tgz ...
>> cub200 seed 1 (PriViLege): starting
python train.py -project base -dataset cub200 -base_mode ft_dot -new_mode avg_cos -gamma 0.1 -lr_base 2e-4 -lr_new 2e-4 -decay 0.0005 -epochs_base 5 -epochs_new 3 -schedule Cosine -milestones 20 30 45 -gpu 0 -temperature 16 -start_session 0 -batch_size_base 128 -seed 1 -vit -comp_out 1 -LT -ED -SKD  -out PriViLege -dataroot /content/data -ckpt_dir /content/drive/MyDrive/PriViLege/resume/PriViLege/cub200/seed1 

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
GPU COUNT 0
Devices: []
Device Name: []
manual seed: 1
{'ED': True,
 'ED_hp': 0.1,
 'LT': True,
 'MP': False,
 'PKT_tune_way': 1,
 'SKD': True,
 'WC': False,
 'base_mode': 'ft_dot',
 'baseline': False,
 'batch_size_base': 128,
 'batch

In [ ]:
prepare('mini_imagenet'); check_layout('mini_imagenet')
for seed in [1, 2, 3, 4, 5]:
    run('mini_imagenet', seed)

miniimagenet.tar is not in Drive: downloading it from the CEC Google Drive folder ...


Downloading...
From (original): https://drive.google.com/uc?id=1_x4o0iFetEv-T3PeIxdSbBPUG4hFfT8U
From (redirected): https://drive.google.com/uc?id=1_x4o0iFetEv-T3PeIxdSbBPUG4hFfT8U&confirm=t&uuid=7d474383-2911-4e3b-a1ae-a462287cc97e
To: /content/miniimagenet.tar
 49%|████▉     | 1.56G/3.19G [00:41<00:41, 38.8MB/s]

## 9. Summary vs. the paper
**Why:** computes A_Base, A_Last and A_Avg exactly as the paper defines them, from the files the code writes. The std is the sample std; the paper doesn't say which it uses.

In [ ]:
summarize('PriViLege')

## 10. Ablations on CUB200 (paper Tables 3 and 4)
**Why:** these test the paper's claims about **why** the method works: each loss should add accuracy, and tuning 2 layers should beat tuning more or none.

Table 3 targets (A_Base / A_Last / A_Avg): PKT 79.06 / 70.81 / 73.36; PKT + L_ED 80.31 / 72.70 / 75.04; PKT + L_SKD 82.10 / 73.44 / 76.27.

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    run('cub200', seed, out='abl_PKT',     losses='')
    run('cub200', seed, out='abl_PKT_ED',  losses='-ED')
    run('cub200', seed, out='abl_PKT_SKD', losses='-SKD')
for out in ['abl_PKT', 'abl_PKT_ED', 'abl_PKT_SKD']:
    summarize(out)

Table 4: number of tuned layers, run without the two losses, as in the paper (its 2-layer row equals the PKT row of Table 3). A_Last targets: 0: 60.19, 2: 70.81, 5: 68.52, 7: 63.15, 10: 57.78, 12: 56.02. The B-Prompt stays in layers 0-1; `-taskblock` only changes how many layers are unfrozen.

In [ ]:
for L in [0, 2, 5, 7, 10, 12]:
    run('cub200', seed=1, out=f'layers_{L}', losses='', extra=f'-taskblock {L}')
for L in [0, 2, 5, 7, 10, 12]:
    summarize(f'layers_{L}')

## 11. Free disk space (optional)
Each run leaves ~0.7 GB of `.pth` checkpoints on the Colab disk. Results are already in Drive, and resume states are in Drive too, so this is safe.

In [ ]:
!find {REPO}/checkpoint -name '*.pth' -delete
!df -h /content | tail -1